<a href="https://colab.research.google.com/github/JoshuneArriaga/Datos-Masivos/blob/main/Tarea_4_5_Datos_Masivos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Instalación del entorno Spark en Google Colab

In [1]:

!pip install -q findspark
!pip install pyspark
!pip install py4j
!pip install -q findspark pyspark py4j mwparserfromhell
import os
import sys

import findspark
findspark.init()
findspark.find()

import pyspark

from pyspark.sql import DataFrame, SparkSession
from typing import List
import pyspark.sql.types as T
import pyspark.sql.functions as F

In [2]:
spark = SparkSession.builder \
    .appName("wikipedia_ml") \
    .getOrCreate()

spark

## Parsing del XML y conversión a RDD
Se descargan 3 partes del dump de la pagina de wikipedia

In [3]:
!mkdir -p data/wikipedia_src
urls = [
"https://dumps.wikimedia.org/enwiki/20260201/enwiki-20260201-pages-articles-multistream1.xml-p1p41242.bz2",
"https://dumps.wikimedia.org/enwiki/20260201/enwiki-20260201-pages-articles-multistream2.xml-p41243p151573.bz2",
"https://dumps.wikimedia.org/enwiki/20260201/enwiki-20260201-pages-articles-multistream3.xml-p151574p311329.bz2"
]

for url in urls:
    !wget -nc {url} -P data/wikipedia_src

File ‘data/wikipedia_src/enwiki-20260201-pages-articles-multistream1.xml-p1p41242.bz2’ already there; not retrieving.



In [27]:
import glob

dump_files = sorted(
    glob.glob("data/wikipedia_src/*.bz2")
)

print("Archivos encontrados:")
for f in dump_files:
    print(f)

Archivos encontrados:
data/wikipedia_src/enwiki-20260201-pages-articles-multistream1.xml-p1p41242.bz2
data/wikipedia_src/enwiki-20260201-pages-articles-multistream2.xml-p41243p151573.bz2
data/wikipedia_src/enwiki-20260201-pages-articles-multistream3.xml-p151574p311329.bz2


Extraer categorías
Label: is_stub
Un "stub" es un artículo marcado como incompleto/corto.
Wikipedia los marca con plantillas como {{stub}}, {{bio-stub}}, etc.
es una clasificación binaria limpia en los datos.

In [30]:
import glob, bz2, re, time
import xml.etree.ElementTree as ET

def detect_namespace(filepath):
    with bz2.open(filepath, 'rt', encoding='utf-8') as f:
        for line in f:
            match = re.search(r'xmlns="([^"]+)"', line)
            if match:
                ns = '{' + match.group(1) + '}'
                print(f'Namespace: {ns}')
                return ns
    return ''

def parse_dump(filepath, max_articles=None):
    NS = detect_namespace(filepath)
    articles = []
    count = 0
    t0 = time.time()

    print('Parseando dump XML...')
    with bz2.open(filepath, 'rt', encoding='utf-8') as f:
        for event, elem in ET.iterparse(f, events=('end',)):
            if elem.tag != f'{NS}page':
                continue

            ns_tag = elem.find(f'{NS}ns')
            if ns_tag is None or ns_tag.text != '0':
                elem.clear(); continue

            revision = elem.find(f'{NS}revision')
            if revision is None:
                elem.clear(); continue

            text_elem = revision.find(f'{NS}text')
            wikitext = text_elem.text if (text_elem is not None and text_elem.text) else ''

            if wikitext.strip().upper().startswith('#REDIRECT'):
                elem.clear(); continue

            title_elem = elem.find(f'{NS}title')
            id_elem    = elem.find(f'{NS}id')
            ts_elem    = revision.find(f'{NS}timestamp')

            categories   = re.findall(r'\[\[Category:([^\]|]+)', wikitext)
            num_links    = len(re.findall(r'\[\[[^:][^\]]*\]\]', wikitext))
            num_refs     = len(re.findall(r'<ref[\s>]', wikitext))
            num_sections = len(re.findall(r'^==[^=]', wikitext, re.MULTILINE))
            num_images   = len(re.findall(r'\[\[(File|Image):', wikitext, re.IGNORECASE))
            num_tables   = len(re.findall(r'\{\|', wikitext))
            is_stub      = 1 if re.search(r'\{\{[^\}]*stub[^\}]*\}\}', wikitext, re.IGNORECASE) else 0
            first_cat    = categories[0].strip() if categories else 'Unknown'

            articles.append({
                'id':             int(id_elem.text) if id_elem is not None else 0,
                'title':          title_elem.text if title_elem is not None else '',
                'word_count':     len(wikitext.split()),
                'text_length':    len(wikitext),
                'timestamp':      ts_elem.text if ts_elem is not None else '',
                'num_categories': len(categories),
                'num_links':      num_links,
                'num_references': num_refs,
                'num_sections':   num_sections,
                'num_images':     num_images,
                'num_tables':     num_tables,
                'first_category': first_cat,
                'is_stub':        is_stub,
            })

            count += 1
            if count % 5000 == 0:
                print(f'  {count:>6,} artículos  ({time.time()-t0:.0f}s)')
            elem.clear()

            if max_articles and count >= max_articles:
                break

    print(f'Terminado: {count:,} artículos  ({time.time()-t0:.0f}s)')
    return articles


In [31]:
dump_files = sorted(glob.glob("data/wikipedia_src/*.bz2"))
df_total = None

for file in dump_files:
    print(f"\n── {file}")
    articles_part = parse_dump(file, max_articles=60000)
    df_part = spark.createDataFrame(articles_part)
    df_part = df_part.withColumn("source_dump", F.lit(file))
    df_total = df_part if df_total is None else df_total.union(df_part)

df = df_total
df.cache()
print(f"\nTotal artículos: {df.count():,}")
df.printSchema()


── data/wikipedia_src/enwiki-20260201-pages-articles-multistream1.xml-p1p41242.bz2
Namespace: {http://www.mediawiki.org/xml/export-0.11/}
Parseando dump XML...
   5,000 artículos  (31s)
  10,000 artículos  (64s)
  15,000 artículos  (99s)
  20,000 artículos  (127s)
Terminado: 21,009 artículos  (130s)

── data/wikipedia_src/enwiki-20260201-pages-articles-multistream2.xml-p41243p151573.bz2
Namespace: {http://www.mediawiki.org/xml/export-0.11/}
Parseando dump XML...
   5,000 artículos  (28s)
  10,000 artículos  (54s)
  15,000 artículos  (80s)
  20,000 artículos  (97s)
  25,000 artículos  (114s)
  30,000 artículos  (130s)
  35,000 artículos  (141s)
  40,000 artículos  (151s)
  45,000 artículos  (158s)
  50,000 artículos  (172s)
  55,000 artículos  (180s)
  60,000 artículos  (190s)
Terminado: 60,000 artículos  (190s)

── data/wikipedia_src/enwiki-20260201-pages-articles-multistream3.xml-p151574p311329.bz2
Namespace: {http://www.mediawiki.org/xml/export-0.11/}
Parseando dump XML...
   5,000 

In [33]:
df.count()

134851

In [34]:
df.groupBy("source_dump").count().show()

+--------------------+-----+
|         source_dump|count|
+--------------------+-----+
|data/wikipedia_sr...|21009|
|data/wikipedia_sr...|60000|
|data/wikipedia_sr...|53842|
+--------------------+-----+



## **Estadísticas descriptivas**

In [35]:
df.select('word_count','text_length','num_categories','num_links',
          'num_references','num_sections','num_images','num_tables').summary().show()

print("\n── Balance de clases ──")
df.groupBy('is_stub').count().withColumn(
    'pct', F.round(F.col('count') / df.count() * 100, 2)
).orderBy('is_stub').show()

print("\n Comparación stubs vs no-stubs")
df.groupBy('is_stub').agg(
    F.round(F.avg('word_count'), 0).alias('avg_words'),
    F.round(F.avg('num_links'), 1).alias('avg_links'),
    F.round(F.avg('num_references'), 1).alias('avg_refs'),
    F.round(F.avg('num_sections'), 1).alias('avg_sections'),
    F.round(F.avg('num_images'), 1).alias('avg_images'),
).orderBy('is_stub').show()

+-------+------------------+-----------------+-----------------+------------------+-----------------+------------------+------------------+------------------+
|summary|        word_count|      text_length|   num_categories|         num_links|   num_references|      num_sections|        num_images|        num_tables|
+-------+------------------+-----------------+-----------------+------------------+-----------------+------------------+------------------+------------------+
|  count|            134851|           134851|           134851|            134851|           134851|            134851|            134851|            134851|
|   mean|3399.3656183491407|29617.45498364862| 7.27251559128223| 152.7055787498795|41.71071775515198|7.4626662019562335| 4.056529057997345|0.6705771555272115|
| stddev| 4163.536594049936| 38008.1032541333|9.330968895590933|212.25184122935465|69.04591794286392| 4.064811231137456|11.387613068496032| 2.499879194678524|
|    min|                10|               94|

## **PREPROCESAMIENTO ML**

In [36]:
from pyspark.ml.feature import VectorAssembler, StandardScaler, ChiSqSelector
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml import Pipeline


## Eliminar nulos

In [37]:
df_clean = df.dropna(subset=['word_count','num_links','num_references',
                              'num_sections','num_images','num_tables','num_categories'])
print(f"Registros tras dropna: {df_clean.count():,}")

Registros tras dropna: 134,851


## Features

In [38]:
feature_cols = [
    'word_count',
    'text_length',
    'num_categories',
    'num_links',
    'num_references',
    'num_sections',
    'num_images',
    'num_tables',
]

assembler = VectorAssembler(inputCols=feature_cols, outputCol='features')
scaler    = StandardScaler(inputCol='features', outputCol='features_scaled',
                           withMean=True, withStd=True)



In [39]:
# Selección de características con Chi-Square
selector = ChiSqSelector(
    numTopFeatures=6,
    featuresCol='features_scaled',
    labelCol='is_stub',
    outputCol='features_selected'
)

# Split train/test
train, test = df_clean.randomSplit([0.7, 0.3], seed=42)
print(f"Train: {train.count():,}  |  Test: {test.count():,}")


Train: 94,482  |  Test: 40,369


## MODELO 1 — REGRESIÓN LOGÍSTICA

In [40]:
# Calcular pesos
total         = df_clean.count()
stub_count    = df_clean.filter(F.col('is_stub') == 1).count()
no_stub_count = df_clean.filter(F.col('is_stub') == 0).count()

weight_stub    = total / (2 * stub_count)
weight_no_stub = total / (2 * no_stub_count)

df_clean = df_clean.withColumn(
    'class_weight',
    F.when(F.col('is_stub') == 1, weight_stub).otherwise(weight_no_stub)
)

train, test = df_clean.randomSplit([0.7, 0.3], seed=42)

In [41]:
# Agregar weightCol a los modelos
pipeline_lr = Pipeline(stages=[assembler, scaler,
    LogisticRegression(
        featuresCol='features_scaled',
        labelCol='is_stub',
        weightCol='class_weight',   # <--
        maxIter=100,
        regParam=0.01,
    )
])

model_lr   = pipeline_lr.fit(train)
preds_lr   = model_lr.transform(test)

eval_bin  = BinaryClassificationEvaluator(labelCol='is_stub', metricName='areaUnderROC')
eval_acc  = MulticlassClassificationEvaluator(labelCol='is_stub', predictionCol='prediction', metricName='accuracy')
eval_f1   = MulticlassClassificationEvaluator(labelCol='is_stub', predictionCol='prediction', metricName='f1')

auc_lr = eval_bin.evaluate(preds_lr)
acc_lr = eval_acc.evaluate(preds_lr)
f1_lr  = eval_f1.evaluate(preds_lr)

print(f"  AUC-ROC  : {auc_lr:.4f}")
print(f"  Accuracy : {acc_lr:.4f}")
print(f"  F1-Score : {f1_lr:.4f}")

# Matriz de confusión manual
print("\n  Matriz de confusión (LR):")
preds_lr.groupBy('is_stub', 'prediction').count().orderBy('is_stub','prediction').show()


  AUC-ROC  : 0.8217
  Accuracy : 0.7034
  F1-Score : 0.7927

  Matriz de confusión (LR):
+-------+----------+-----+
|is_stub|prediction|count|
+-------+----------+-----+
|      0|       0.0|27054|
|      0|       1.0|11656|
|      1|       0.0|  318|
|      1|       1.0| 1341|
+-------+----------+-----+



In [42]:
from pyspark.ml.functions import vector_to_array

# Ver la distribución de probabilidades predichas
preds_lr.select(
    'is_stub',
    vector_to_array('probability')[1].alias('prob_stub')
).groupBy('is_stub').agg(
    F.round(F.avg('prob_stub'), 3).alias('avg_prob_stub'),
    F.round(F.min('prob_stub'), 3).alias('min_prob_stub'),
    F.round(F.max('prob_stub'), 3).alias('max_prob_stub'),
).show()

# Probar threshold más alto (default es 0.5, subir a 0.6-0.7)
lr_model = model_lr.stages[-1]
lr_model.setThreshold(0.65)

preds_lr_t = model_lr.transform(test)

print("Con threshold=0.65:")
preds_lr_t.groupBy('is_stub', 'prediction').count().orderBy('is_stub','prediction').show()

+-------+-------------+-------------+-------------+
|is_stub|avg_prob_stub|min_prob_stub|max_prob_stub|
+-------+-------------+-------------+-------------+
|      0|        0.367|          0.0|        0.999|
|      1|        0.637|        0.006|        0.874|
+-------+-------------+-------------+-------------+

Con threshold=0.65:
+-------+----------+-----+
|is_stub|prediction|count|
+-------+----------+-----+
|      0|       0.0|34328|
|      0|       1.0| 4382|
|      1|       0.0|  606|
|      1|       1.0| 1053|
+-------+----------+-----+



## MODELO 2 — RANDOM FOREST

In [43]:
pipeline_rf = Pipeline(stages=[assembler, scaler,
    RandomForestClassifier(
        featuresCol='features_scaled',
        labelCol='is_stub',
        weightCol='class_weight',
        numTrees=100,
        maxDepth=10,
        seed=42,
    )
])

model_rf  = pipeline_rf.fit(train)
preds_rf  = model_rf.transform(test)

auc_rf = eval_bin.evaluate(preds_rf)
acc_rf = eval_acc.evaluate(preds_rf)
f1_rf  = eval_f1.evaluate(preds_rf)

print(f"  AUC-ROC  : {auc_rf:.4f}")
print(f"  Accuracy : {acc_rf:.4f}")
print(f"  F1-Score : {f1_rf:.4f}")

print("\n  Matriz de confusión (RF):")
preds_rf.groupBy('is_stub', 'prediction').count().orderBy('is_stub','prediction').show()

# Importancia de features
rf_model = model_rf.stages[-1]
importances = list(zip(feature_cols, rf_model.featureImportances.toArray()))
importances.sort(key=lambda x: x[1], reverse=True)
print("\n  Importancia de features:")
for name, imp in importances:
    print(f"    {name:<20} {imp:.4f}")

  AUC-ROC  : 0.9331
  Accuracy : 0.9344
  F1-Score : 0.9457

  Matriz de confusión (RF):
+-------+----------+-----+
|is_stub|prediction|count|
+-------+----------+-----+
|      0|       0.0|36420|
|      0|       1.0| 2290|
|      1|       0.0|  357|
|      1|       1.0| 1302|
+-------+----------+-----+


  Importancia de features:
    text_length          0.2708
    word_count           0.2642
    num_links            0.1590
    num_categories       0.1384
    num_sections         0.0949
    num_references       0.0401
    num_images           0.0227
    num_tables           0.0099
